# Guardrail 10 — PII Detection & Redaction

**Where it sits:** at *every* boundary — input, prompt assembly, retrieved-chunk scrub, output, audit-log write. PII is not a one-place fix.

**What it stops:** SSN, credit card numbers, emails, phone numbers, IP addresses, and credentials from leaking to (a) other users, (b) the audit log, (c) downstream systems, (d) the LLM provider's logs.

**Why a dedicated guard:** I touched PII in notebooks 1 (audit-log scrub) and 7 (response redaction). That's not enough. Industry treats PII as its own layer with its own threat model:
  - some PII gets redacted in the response but kept in the audit log (forensics)
  - some gets redacted in the audit log but kept in the response (the user needs their own data)
  - some gets blocked entirely (a credit card number in a query is almost never legitimate)

**Decision contract:** `{allow | rewrite | block, sanitized_text, entities[], reasons[]}`

**Self-contained:** inlines a toy entity detector. No imports from other folders.

## Step 1 — toy PII detector

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import re

PII_PATTERNS = [
    ("SSN",          re.compile(r"\b\d{3}-\d{2}-\d{4}\b")),
    ("CREDIT_CARD",  re.compile(r"\b(?:\d[ -]?){13,16}\b")),
    ("EMAIL",        re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")),
    ("PHONE_US",     re.compile(r"\b(?:\+?1[-. ]?)?\(?\d{3}\)?[-. ]?\d{3}[-. ]?\d{4}\b")),
    ("IPV4",         re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")),
    ("API_KEY",      re.compile(r"\b(sk-[A-Za-z0-9]{20,}|ghp_[A-Za-z0-9]{20,}|AKIA[0-9A-Z]{16})\b")),
]

def detect_pii(text: str):
    """Return a list of (entity_type, value, span) tuples."""
    hits = []
    for label, pat in PII_PATTERNS:
        for m in pat.finditer(text):
            hits.append({"type": label, "value": m.group(), "span": m.span()})
    return sorted(hits, key=lambda h: h["span"][0])

print("PII detector loaded with patterns:", [t[0] for t in PII_PATTERNS])

## Step 2 — redaction strategies

In [ ]:
REDACT_FULL   = lambda v, t: f"[{t}]"                    # "[SSN]"
REDACT_HASH   = lambda v, t: f"[{t}:{hash(v) & 0xFFFF:04x}]"  # "[SSN:a3f2]" — correlatable, not reversible
REDACT_PARTIAL = {
    "SSN":         lambda v: f"***-**-{v[-4:]}",
    "CREDIT_CARD": lambda v: f"****-****-****-{v[-4:]}",
    "EMAIL":       lambda v: v.split("@")[0][:2] + "***@" + v.split("@")[1],
    "PHONE_US":    lambda v: f"***-***-{v[-4:]}",
    "IPV4":        lambda v: ".".join(["***"] * 3) + "." + v.split(".")[-1],
    "API_KEY":     lambda v: v[:6] + "..." + v[-4:],
}
REDACT_PARTIAL["SSN"] = lambda v: f"***-**-{v[-4:]}"

def redact(text: str, hits: list, strategy: str = "partial"):
    """Replace each PII span with the chosen strategy's output.
    Strategies: 'full' (remove entirely), 'hash' (correlatable),
    'partial' (show last-4 for forensic value)."""
    if not hits:
        return text
    out, cursor = [], 0
    for h in hits:
        out.append(text[cursor:h["span"][0]])  # keep the gap before
        v, t = h["value"], h["type"]
        if strategy == "full":
            out.append(REDACT_FULL(v, t))
        elif strategy == "hash":
            out.append(REDACT_HASH(v, t))
        elif strategy == "partial":
            fn = REDACT_PARTIAL.get(t)
            out.append(fn(v) if fn else f"[{t}]")
        cursor = h["span"][1]
    out.append(text[cursor:])
    return "".join(out)

## Step 3 — the differential: where each strategy belongs

In [ ]:
sample = ("Contact me at alice@example.com or 555-123-4567. "
          "My SSN is 123-45-6789 and the card is 4111-1111-1111-1111. "
          "API key sk-abcdefghijklmnopqrstuv from server 10.0.0.5.")
hits = detect_pii(sample)

print("DETECTED ENTITIES:")
for h in hits:
    print(f"  {h['type']:12s} → {h['value']!r}  at {h['span']}")

print("\nREDACTION STRATEGIES — where each belongs:")
for strat in ("full", "partial", "hash"):
    print(f"\n  [{strat:7s}] {redact(sample, hits, strat)}")

print("""
  Where each strategy lives in the pipeline:
    input guard  →  KEEP in query (user needs their question answered)
                   HASH for audit log (correlatable, not raw)
    output guard →  PARTIAL (show last-4 if the value is the user's own)
                   FULL if the value belongs to someone else
    audit log    →  HASH or FULL (never raw PII on disk)
    LLM provider →  FULL (their logs are outside your control)
    downstream   →  whatever the contract specifies (often FULL)
""")

## Step 4 — the full PII guard

In [ ]:
def pii_guard(text: str,
              boundary: str,
              block_severity: tuple = ("SSN", "CREDIT_CARD", "API_KEY")):
    """
    boundary: 'input' | 'output' | 'audit_log' | 'llm_provider'
    """
    hits = detect_pii(text)
    if not hits:
        return {"decision": "allow", "sanitized": text, "entities": [], "reasons": []}

    types_present = {h["type"] for h in hits}

    # Some PII types are always a block, regardless of boundary
    if any(t in block_severity for t in types_present) and boundary == "input":
        return {"decision": "block",
                "sanitized": None,
                "entities": hits,
                "reasons": [f"blocked_pii_in_input:{types_present & set(block_severity)}"]}

    # Pick the strategy based on where we are in the pipeline
    strategy = {
        "input":       "partial",   # user keeps their own data visible
        "output":      "partial",
        "audit_log":   "hash",      # correlatable, never raw
        "llm_provider":"full",      # outside our control
    }.get(boundary, "full")

    return {"decision": "rewrite",
            "sanitized": redact(text, hits, strategy),
            "entities": hits,
            "reasons": [f"redacted:{len(hits)}_via_{strategy}"]}

In [ ]:
tests = [
    ("user query",         sample,                    "input"),
    ("model response",     sample,                    "output"),
    ("audit log entry",    sample,                    "audit_log"),
    ("prompt to provider", sample,                    "llm_provider"),
    ("SSN in input → block", "My SSN is 123-45-6789, what's the capital of France?", "input"),
    ("clean text",         "What is the capital of France?", "output"),
]

for label, text, boundary in tests:
    r = pii_guard(text, boundary)
    print(f"\n=== {label} (boundary={boundary}) ===")
    print(f"  decision: {r['decision']}")
    print(f"  reasons : {r['reasons']}")
    if r['sanitized'] is not None:
        print(f"  sanitized: {r['sanitized'][:90]}{'...' if len(r['sanitized'])>90 else ''}")

## Step 5 — the threat model

In [ ]:
print("""
PII leaks happen at five distinct boundaries, and a fix at one does
not cover the others:

  1. USER  →  ENGINE     (input)
     User pastes their own SSN. Engine must accept the question
     but not write the SSN raw to the audit log.

  2. ENGINE  →  LLM PROVIDER
     Provider's logs are outside your control. Redact before sending.

  3. LLM  →  ENGINE      (output)
     Model regurgitates PII it saw in retrieved docs. Redact before
     the user sees it.

  4. ENGINE  →  AUDIT LOG
     Even internal logs leak. Hash or full-redact.

  5. ENGINE  →  DOWNSTREAM
     Other systems (analytics, BI, customer support tools). Redact
     per the downstream contract.

Industry tools:
  • Microsoft Presidio  — open-source PII detector + anonymizer
  • AWS Comprehend PII  — managed detection
  • Google Cloud DLP    — managed detection + de-identification
  • Gretel, Tonic        — synthetic-data alternatives
  • Regex alone misses   — names, addresses, custom IDs. ML-based
    detectors catch what regex doesn't.
""")

In [ ]:
### Real LangChain demo: PII guard as a Runnable before sending to the LLM

from langchain_core.runnables import RunnableLambda

def _redact_for_provider(text: str):
    """Full-redact PII before the text leaves your trust boundary."""
    r = pii_guard(text, boundary="llm_provider")
    return r["sanitized"] or ""

redact_runnable = RunnableLambda(_redact_for_provider)
print(f"PII redactor constructed: {redact_runnable}")
print(f"sample: {redact_runnable.invoke('My SSN is 123-45-6789, email alice@example.com.')}")


## Takeaways

- **PII is a multi-boundary problem.** A single redaction step at the response covers *one* leak. Real systems have one at every boundary.
- **Strategy is not 'redact.'** It's 'redact differently depending on where this text is going.' Full for the LLM provider, partial for the user, hash for the audit log.
- **Block at input for severity-1 types.** A credit card number pasted into a chat is almost never legitimate — block and ask the user to use the payment portal instead.
- **Regex is the floor, not the ceiling.** SSN, credit card, email, phone — these are catchable by regex. Names, addresses, custom IDs require an ML detector. Industry uses Presidio / Comprehend / DLP for the ML layer.
- **The LLM provider is outside your trust boundary.** Their logs, their retention, their subpoena surface. Treat the prompt you send them like you'd treat a postcard.
- **HIPAA / GDPR / PCI-DSS each have their own PII rules.** A 'PII guard' that's not configurable per regulation isn't a guard, it's a guess.

**Negative fixture checklist:** SSN in query, credit card in output, API key in prompt, PII in audit log, PII that *belongs to someone else* in retrieved chunk. ✓